# Flaky Test Classification using Gemini

This notebook evaluates Google's Gemini models for flaky test identification and classification.

The workflow follows the same evaluation pipeline used for local LLMs to ensure comparable results.

In [1]:
!pip install -q google-genai pandas tqdm


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
import json
import time
from pathlib import Path

import pandas as pd
from tqdm import tqdm

from google import genai

In [ ]:
# ============================================
# Gemini Configuration
# ============================================
from dotenv import load_dotenv
import os

load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

MODEL_NAME = "gemini-3.1-flash-lite"

PROMPT_TYPE = "few_shot_cot"
# zero_shot
# zero_shot_cot
# few_shot_cot

USE_CONTEXT = True

TEMPERATURE = 0

### initialize gemin client

In [31]:
client = genai.Client(
    api_key=GEMINI_API_KEY
)

### project paths

In [21]:
PROJECT_ROOT = Path.cwd().parent

DATASET_PATH = (
    PROJECT_ROOT
    / "datasets"
    / "evaluation_dataset.jsonl"
)

RESULT_ROOT = (
    PROJECT_ROOT
    / "results"
)

### import prompt builder

In [23]:
import sys

sys.path.insert(0, str(PROJECT_ROOT))

from utils.prompt_builder import build_prompt

In [24]:
dataset = pd.read_json(
    DATASET_PATH,
    lines=True
)

print(f"Loaded {len(dataset)} samples")

dataset.head()

Loaded 2210 samples


,id,test_id,isFlaky,issue_category,repo_url,issue_commit,fixed_commit,test_code,helper_methods_json,failure_log,code_under_test_json,test_code_original,helper_methods_json_original,failure_log_original,code_under_test_original,has_helper_methods,has_code_under_test,has_failure_log,context_score
0,857,ormlitecore59309e55,True,Order Dependent,https://github.com/j256/ormlite-core,59309e51c61e8a63cb5fd24a5a7607b668a5f095,c80bde196ca152ecc8a3c4f38f77dbe5a4ea3232,@Test\n\tpublic void testSetObjectCacheThrow()...,{},org.opentest4j.AssertionFailedError: Unexpecte...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,@Test\n\tpublic void testSetObjectCacheThrow()...,{},Failed Rounds: 10/10\norg.opentest4j.Assertion...,{'com.j256.ormlite.dao.BaseDaoImpl': {'setObje...,False,True,True,3
1,1574,Closure-144-21,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.jscomp.Result': {'<ini...,public void testExportMultiple3() throws Excep...,{'compileAndCheck': 'private void compileAndCh...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.jscomp.Result': {'<ini...,True,True,True,5
2,1483,Closure-115-5,False,Non-Flaky,https://github.com/google/closure-compiler,2d6e1c78f41248fbbb1eec43b23e7430e2bc7885,4597738e8898f738c1f969fe90479728be81cc80,public void testInlineFunctions6() {\n\n te...,"{'test': 'public void test(String js, String e...",junit.framework.AssertionFailedError:\nExpecte...,{'com.google.javascript.jscomp.DiagnosticType'...,public void testInlineFunctions6() {\n // m...,{'test': '/** * Verifies that the compiler ...,Failed Rounds: 1/1\njunit.framework.AssertionF...,{'com.google.javascript.jscomp.DiagnosticType'...,True,True,True,5
3,431,ignite3modulesstoragerocksdb19c8a82testAbortWrite,True,Implementation Dependent,https://github.com/apache/ignite-3,19c8a824bd9d31f0d0dbd3fbbdd2a32ee360cab2,de6ee0702398f9ce3022a8e265c633857f3a3d88,.\n */\n @Test\n public void testAbo...,{'read': 'protected BinaryRow read(RowId rowId...,org.junit.jupiter.api.extension.ParameterResol...,{'org.apache.ignite.internal.hlc.HybridTimesta...,.\n */\n @Test\n public void testAbo...,{'read': '/** * Reads a row. */ ...,Failed Rounds: 84/101\norg.junit.jupiter.api.e...,{'org.apache.ignite.internal.hlc.HybridTimesta...,True,True,True,5
4,1563,Closure-144-10,False,Non-Flaky,https://github.com/google/closure-compiler,c9e89727dc8063d087d28e42629606f4fd74a6e5,465282f1ca28a208b06c47b55fd292d4631c55da,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,junit.framework.ComparisonFailure: expected:<....,{'com.google.javascript.rhino.Node': {'getDoub...,public void testVariableArgumentsTypesAnnotati...,{'assertTypeAnnotations': 'private void assert...,Failed Rounds: 1/1\njunit.framework.Comparison...,{'com.google.javascript.rhino.Node': {'getDoub...,True,True,True,5


In [25]:
sample = dataset.iloc[0].to_dict()

prompt = build_prompt(
    sample=sample,
    strategy=PROMPT_TYPE,
    include_context=USE_CONTEXT
)

print(prompt)

You are an expert software testing assistant specializing in flaky test identification and classification.

Base every decision solely on the provided artifacts.

Do not rely on external knowledge or assumptions.

If the available evidence is insufficient to confidently identify a flaky test, classify it as Non-Flaky.

Your task is to analyze the provided software testing artifacts.

Determine whether the test is Flaky or Non-Flaky.

If the test is Flaky, select exactly one issue category from the provided category list.

Justify the classification using only the provided evidence.

Identify the artifacts that support your conclusion.

## Flaky Test Categories

If the test is classified as Flaky, select exactly one issue category from the following list.

1. Implementation Dependent
   - The test outcome depends on implementation-specific behavior rather than the intended specification.

2. Order Dependent
   - The test outcome depends on the execution order of tests because of shared 

### gemini inference function

In [32]:
import json
import time

def classify_prompt(prompt):

    start = time.perf_counter()

    try:

        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        latency = (time.perf_counter() - start) * 1000

        raw = response.text.strip()

        # Remove markdown fences if present
        if raw.startswith("```"):
            raw = (
                raw.replace("```json", "")
                   .replace("```", "")
                   .strip()
            )

        prediction = json.loads(raw)

        return {
            "result": prediction,
            "model": MODEL_NAME,
            "latency_ms": round(latency, 2),
            "raw_response": response.text
        }

    except Exception as e:

        return {
            "result": {
                "classification": "ERROR",
                "category": "ERROR",
                "reasoning": str(e),
                "evidence": []
            },
            "model": MODEL_NAME,
            "latency_ms": None,
            "raw_response": str(e)
        }

### test gemini communication

In [33]:
test_prompt = """
Return ONLY this JSON.

{
"classification":"Non-Flaky",
"category":"Not Applicable",
"reasoning":"Test",
"evidence":["Test Code"]
}
"""

response = classify_prompt(test_prompt)

print(response)

{'result': {'classification': 'Non-Flaky', 'category': 'Not Applicable', 'reasoning': 'Test', 'evidence': ['Test Code']}, 'model': 'gemini-3.1-flash-lite', 'latency_ms': 2595.15, 'raw_response': '{\n"classification":"Non-Flaky",\n"category":"Not Applicable",\n"reasoning":"Test",\n"evidence":["Test Code"]\n}'}


In [28]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-3.6-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-

### Evaluate Dataset

In [151]:
print(dataset.columns.tolist())

['id', 'test_id', 'isFlaky', 'issue_category', 'repo_url', 'issue_commit', 'fixed_commit', 'test_code', 'helper_methods_json', 'failure_log', 'code_under_test_json', 'test_code_original', 'helper_methods_json_original', 'failure_log_original', 'code_under_test_original', 'has_helper_methods', 'has_code_under_test', 'has_failure_log', 'context_score']


In [152]:
sample = dataset.iloc[0].to_dict()

sample

{'id': 857,
 'test_id': 'ormlitecore59309e55',
 'isFlaky': True,
 'issue_category': 'Order Dependent',
 'repo_url': 'https://github.com/j256/ormlite-core',
 'issue_commit': '59309e51c61e8a63cb5fd24a5a7607b668a5f095',
 'fixed_commit': 'c80bde196ca152ecc8a3c4f38f77dbe5a4ea3232',
 'test_code': '@Test\n\tpublic void testSetObjectCacheThrow() throws Exception {\n\t\t@SuppressWarnings("unchecked")\n\t\tDao<Foo, String> dao = (Dao<Foo, String>) createMock(Dao.class);\n\t\tRuntimeExceptionDao<Foo, String> rtDao = new RuntimeExceptionDao<Foo, String>(dao);\n\t\tdao.setObjectCache(false);\n\t\texpectLastCall().andThrow(new SQLException("Testing catch"));\n\t\treplay(dao);\n\t\tassertThrowsExactly(RuntimeException.class, () -> {\n\t\t\trtDao.setObjectCache(false);\n\t\t});\n\t\tverify(dao);\n\t}',
 'helper_methods_json': {},
 'failure_log': 'org.opentest4j.AssertionFailedError: Unexpected exception type thrown, expected: <java.lang.RuntimeException> but was: <java.lang.AssertionError>\n\tat org.j

### Evaluation Loop

In [153]:
results = []

for _, row in tqdm(
    dataset.iterrows(),
    total=len(dataset),
    desc="Evaluating"
):

    sample = row.to_dict()

    prompt = build_prompt(
        sample=sample,
        strategy=PROMPT_TYPE,
        include_context=USE_CONTEXT
    )

    response = classify_prompt(prompt)

    prediction = response["result"]

    record = {

        "id": sample["id"],

        "test_id": sample["test_id"],

        "ground_truth_classification":
            "Flaky"
            if sample["isFlaky"]
            else "Non-Flaky",

        "ground_truth_category":
            sample["issue_category"],

        "predicted_classification":
            prediction["classification"],

        "predicted_category":
            prediction["category"],

        "reasoning":
            prediction["reasoning"],

        "evidence":
            prediction["evidence"],

        "model":
            response["model"],

        "prompt_type":
            PROMPT_TYPE,

        "context_enabled":
            USE_CONTEXT,

        "latency_ms":
            response["latency_ms"]

    }

    results.append(record)

print(f"Finished {len(results)} predictions")

Evaluating: 100%|██████████| 2210/2210 [04:31<00:00,  8.14it/s]

Finished 2210 predictions


### Inspect Results

In [154]:
results[0]

{'id': 857,
 'test_id': 'ormlitecore59309e55',
 'ground_truth_classification': 'Flaky',
 'ground_truth_category': 'Order Dependent',
 'predicted_classification': 'ERROR',
 'predicted_category': 'ERROR',
 'reasoning': "429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \\n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-3.5-flash\\nPlease retry in 26.187754314s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': '

### save results

In [155]:
output_dir = (
    RESULT_ROOT
    / MODEL_NAME
    / PROMPT_TYPE
    / (
        "with_context"
        if USE_CONTEXT
        else "without_context"
    )
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

In [156]:
output_file = output_dir / "predictions.jsonl"

with output_file.open(
    "w",
    encoding="utf-8"
) as f:

    for record in results:

        f.write(
            json.dumps(
                record,
                ensure_ascii=False
            ) + "\n"
        )

print(f"Saved {len(results)} predictions")
print(output_file)

Saved 2210 predictions
d:\university works\Final-Year_Firts_sem\FYP\INFO\REPO\CA-Classification-Framework\results\gemini-3.5-flash\zero_shot_cot\without_context\predictions.jsonl
